**Note (policy graph update):** the pipeline now reads `gmsjourney/policy/charge_review_kg.json` and trains the GMS on it, state layer included, so the first run rebuilds the store; outputs saved below are from the earlier run and refresh on re-execution.

**Earlier executed copy** (2026-10-11): run end-to-end on the **knowlytix** backend — Python 3.12, licence key at `~/.knowlytix/license.key`, real HuggingFace encoders (hub reachable), ~39 min total on the first build; a verification re-run on the cached store (same day) finished in 7.9 min with bit-identical results. CELL 1 and CELL 14 below are adapted so the notebook also runs end-to-end on the built-in open baseline when the knowlytix family or a licence is unavailable.

# Charge-review voice bot — GMS-grounded journey testing

The method of *Beyond "Ship and Pray"* applied to the charge-review journey, on the **knowlytix** GMS
(`pip install knowlytix`, Python 3.12, runtime licence) and the open **proofloop / forgeloop** packages
from the book's companion repository. What those packages do not cover for a multi-turn voice bot —
the journey oracle, the covering-array bases, the realiser/reader and the invariance proof, the
reference bot and its mutants — is implemented in `gmsjourney`. Without knowlytix the same notebook
runs on the built-in open baseline.

| Cell | What it does | Comes from |
|---|---|---|
| 1 | setup + licence (env / Colab secret, never printed) | |
| 2 | the policy graph: the PDF as clean head / relation / tail triples with dialog states, transitions and per-state policy (`policy/charge_review_kg.json`, the source every later step reads) | ours |
| 3 | build or load the GMS: GEODE loop, embed loop (v SFT), contradiction SFT (u), cap-trained store, GMSJudge calibration | knowlytix |
| 4 | primitives + oracle property | knowlytix (ENM, score_triple, cap_radius, tension, holonomy) |
| 5 | policy-question bases, two-way admission, substrate generators | knowlytix QuestionGenerator + ours |
| 6 | journey bases: covering array + closure | ours |
| 7 | official catalogs + application layer | proofloop catalogs + ours |
| 8 | design + comparison with knowlytix DesignMatrix | ours (φp) vs knowlytix |
| 9 | invariance proof | ours |
| 10 | runtime gates calibrated (Appendix C) | forgeloop GMS gates on the knowlytix store |
| 11 | run + attribute | knowlytix DOEAnalyzer + ours |
| 11b | turn by turn: the oracle's expected bot response (tools with arguments and results, bot acts) next to the bot's actual response, per customer turn | ours |
| 12-13 | resilience, mutation (+ runtime monitor) | ours |
| 14 | training corpora | proofloop emitters |
| 15 | one call: full campaign + HTML report | |
| 16 | the exported dataset (`output/campaign_out/dataset/*.csv`) and the trained GMS's canonicals (`gms_canonicals.json`) | ours |


In [1]:
# CELL 1 — setup. knowlytix-core needs Python 3.12 + a runtime licence; any Python >= 3.10 runs the
# built-in open baseline. Upload gmsjourney.zip first when running on Colab.
import os, sys, subprocess
if os.path.exists("gmsjourney.zip") and not os.path.exists("gmsjourney"):
    subprocess.run(["unzip", "-q", "gmsjourney.zip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "scipy", "pandas", "pyyaml"], check=True)
# Optional GMS backend: install what this environment can and fall back to the built-in baseline
# for the rest. (knowlytix ships cp312 wheels only, so the loop also degrades gracefully on 3.10.)
for _pkg in ("knowlytix==1.5.0", "proofloop>=0.1.5", "forgeloop>=0.1.4"):
    _r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
    if _r.returncode != 0:
        print(f"optional package {_pkg.split('==')[0].split('>=')[0]} not installable here; "
              f"the built-in open baseline will be used for it")
sys.path.insert(0, os.path.abspath("gmsjourney") if os.path.exists("gmsjourney/gmsjourney") else os.getcwd())

# Licence: read from the environment, ~/.knowlytix/license.key, or a Colab secret named KNOWLYTIX_LICENSE_KEY.
if not os.environ.get("KNOWLYTIX_LICENSE_KEY"):
    try:
        from google.colab import userdata
        os.environ["KNOWLYTIX_LICENSE_KEY"] = userdata.get("KNOWLYTIX_LICENSE_KEY")
    except Exception:
        pass
from gmsjourney import backend
print("knowlytix backend available:", backend.available())   # the key itself is never printed

knowlytix backend available: True


In [2]:
# CELL 2 — the policy graph: the operating-procedure PDF as head / relation / tail triples, with dialog states
# (trajectory) and the policy that holds in each state. Built once by
#   python -m gmsjourney.knowledge.policy_kg build --pdf fictional_poc_operating_procedure_gpt_readable.pdf
# and read by every later step (GMS training, oracle limits, state-aware ontology mapper, exports).
import pandas as pd
from gmsjourney.pipeline import POLICY
from gmsjourney.knowledge.policy_kg import PolicyKG
kg = PolicyKG.load(POLICY)
print(kg.summary())
STATE = "PRESENT_TXN"                      # any of kg.states
print("states:", ", ".join(kg.states))
display(pd.DataFrame([{k: t.get(k) for k in ("head", "relation", "tail", "kind", "source", "evidence")}
                      for t in kg.state_triples(STATE)]))
display(pd.DataFrame(kg.transitions(STATE)))

{'triples': 720, 'by_kind': {'policy': 284, 'trajectory': 85, 'vocabulary': 88, 'domain': 17, 'state': 246}, 'gms_triples': 509, 'states': 19, 'prompt_states': 7, 'transitions': 64, 'rules': 33, 'sources': ['lexicon', 'pdf.1', 'pdf.2A', 'pdf.2B', 'pdf.2C', 'pdf.2D', 'pdf.2E', 'pdf.2F', 'pdf.3', 'pdf.4', 'pdf.5', 'pdf.6', 'pdf.7', 'program'], 'sha256': '1da208c1f378f2f9', 'document': 'fictional_poc_operating_procedure_gpt_readable.pdf'}
states: OPENING, ASK_AUTH, ASK_AUTH_RETRY, PRESENT_TXN, ANSWER_KB, ASK_CLARIFY, PROVIDE_CONTEXT, CONFIRM_RECORDED, DISCLOSE_FEE, CONFIRM_ACTION, ACTION_DONE, ACK_DECLINED, INFORM_NO_CARD_ACTION, EXPLAIN_LIMIT, EXPLAIN_NO_TXNS, SUMMARIZE, ASK_MORE_HELP, TRANSFER, END


,head,relation,tail,kind,source,evidence
0,PRESENT_TXN,is_a,dialog_state,state,pdf.2C,Ask whether the customer recognizes the transa...
1,PRESENT_TXN,in_stage,capture,state,pdf.2C,Ask whether the customer recognizes the transa...
2,PRESENT_TXN,prompts_customer,yes,state,pdf.2C,Ask whether the customer recognizes the transa...
3,PRESENT_TXN,accepts_act,INFORM_DISPOSITION,state,pdf.2C,Ask whether the customer recognizes the transa...
4,PRESENT_TXN,accepts_act,REQUEST_CARD_ACTION,state,pdf.2C,Ask whether the customer recognizes the transa...
5,PRESENT_TXN,accepts_act,REQUEST_HUMAN,state,pdf.2F,"If the customer requests a human at any point,..."
6,PRESENT_TXN,accepts_act,ASK_KB,state,pdf.3,General policy questions may be answered from ...
7,PRESENT_TXN,accepts_act,UNCLEAR,state,pdf.3,If the bot cannot understand required informat...
8,PRESENT_TXN,interprets_yes_as,RECOGNIZED,state,pdf.2C,Ask whether the customer recognizes the transa...
9,PRESENT_TXN,interprets_no_as,UNRECOGNIZED,state,pdf.2C,Ask whether the customer recognizes the transa...


,src,dst,on,when,rules,edges
0,PRESENT_TXN,CONFIRM_RECORDED,INFORM_DISPOSITION,clear RECOGNIZED/UNRECOGNIZED and record_charg...,"[R9, R12]",[W5]
1,PRESENT_TXN,PROVIDE_CONTEXT,INFORM_DISPOSITION,disposition UNSURE (first time),[R10],[]
2,PRESENT_TXN,ASK_CLARIFY,UNCLEAR,answer unclear (first time),[R22],[]
3,PRESENT_TXN,ANSWER_KB,ASK_KB,general policy question,[R25],[]
4,PRESENT_TXN,TRANSFER,INFORM_DISPOSITION,record_charge_response not SUCCESS,[R14],[W11]
5,PRESENT_TXN,TRANSFER,REQUEST_HUMAN,customer asks for a human,[R20],[]
6,PRESENT_TXN,SUMMARIZE,REQUEST_HUMAN,customer asks for a human after responses were...,"[R19, R20]",[]
7,PRESENT_TXN,SUMMARIZE,INFORM_DISPOSITION,record SUCCESS and no transaction pending,"[R12, R19]",[W10]
8,PRESENT_TXN,TRANSFER,any,max_turns reached before the journey completes,[R28],[]


In [3]:
# CELL 3 — build (first run, ~10-45 min on CPU) or load the GMS. Reloads from the store directory afterwards.
from gmsjourney.pipeline import build_gms
store, build_report, backend_info = build_gms(POLICY)
print(backend_info["selected"], backend_info.get("versions"))
if backend_info["selected"] == "knowlytix":
    g = build_report["geode"]
    print("GEODE:", {k: g[k] for k in ("converged", "iterations", "n_proposed", "n_kept_fact_triples")},
          "| dedup proposals rejected:", len(g["duplicates_rejected"]))
    print("encoders:", build_report["encoders"]["kind"], "-", build_report["encoders"]["reason"])
    print("v-space SFT (held-out noisy aliases):", build_report["embed_loop"]["before"], "->", build_report["embed_loop"]["after"])
    print("u-space SFT (held-out phrasings):", build_report["contradiction_sft"]["before"], "->", build_report["contradiction_sft"]["after"])
    print("GMSJudge thresholds:", build_report["judge_thresholds"])
    print("ENM agrees with the compiler:", build_report["enm_agreement"]["all_agree"])
print(store.summary())

[1/12] loading the knowlytix GMS store (~\.cache\gmsjourney\stores\fb6a20bb1e100f0d)


knowlytix {'knowlytix-core': '1.5.0', 'knowlytix-knowledge': '1.5.0', 'knowlytix-harness': '1.5.0', 'knowlytix-benchmark': '1.5.0'}
GEODE: {'converged': True, 'iterations': 3, 'n_proposed': 873, 'n_kept_fact_triples': 608} | dedup proposals rejected: 20
encoders: huggingface - found in the local HF cache
v-space SFT (held-out noisy aliases): 0.45614035087719296 -> 0.7017543859649122
u-space SFT (held-out phrasings): 0.9871794871794872 -> 1.0
GMSJudge thresholds: {'geodesic': 1.0119513763859866, 'path_transport': 1.3434376701340078, 'tension_status': 'degenerate', 'holonomy': 7.010649181008339}
ENM agrees with the compiler: True
{'entities': 135, 'triples': 528, 'relations': 65, 'enm_entries': 12, 'rules': 33, 'workflow_edges': 21, 'geometric_reads_cached': 2111, 'backend': 'knowlytix', 'kx_entities': 229, 'kx_triples': 607, 'kx_enm': 12}


In [4]:
# CELL 4 — primitives, and the oracle property (true facts inside the calibrated cut, substitutions outside).
from gmsjourney.knowledge.geode import oracle_property
print("lookup_enm(limit, max_transactions_per_session) =", store.lookup_enm("limit", "max_transactions_per_session"))
print("score_triple(fee 15) =", round(store.score_triple("expedited_delivery", "has_fee_usd", "15"), 3),
      "| score_triple(fee 0) =", round(store.score_triple("expedited_delivery", "has_fee_usd", "0"), 3),
      "| cut =", round(store.tau("has_fee_usd"), 3))
print("tension(RECOGNIZED, UNRECOGNIZED) =", round(store.tension_energy("RECOGNIZED", "UNRECOGNIZED"), 3))
print("stance(unrecognized is proof of fraud):", store.stance_check("unrecognized_transaction", "is_proof_of", "fraud", +1)["verdict"])
for o in oracle_property(store):
    print(f"  {o['head']:22s} {o['relation']:34s} true {o['score_true']:.3f}  substituted {o['score_fabricated']:.3f}  cut {o['tau']:.3f}  ok={o['calibrated_ok']}")

lookup_enm(limit, max_transactions_per_session) = 4
score_triple(fee 15) = 0.075 | score_triple(fee 0) = 1.055 | cut = 0.297
tension(RECOGNIZED, UNRECOGNIZED) = 1.035
stance(unrecognized is proof of fraud): contradicted
  expedited_delivery     has_business_days_max              true 0.097  substituted 1.139  cut 0.334  ok=True
  standard_delivery      has_business_days_max              true 0.089  substituted 1.146  cut 0.334  ok=True
  expedited_delivery     has_business_days_min              true 0.251  substituted 1.084  cut 0.306  ok=True
  standard_delivery      has_business_days_min              true 0.239  substituted 1.120  cut 0.306  ok=True
  expedited_delivery     has_fee_usd                        true 0.075  substituted 1.055  cut 0.297  ok=True
  standard_delivery      has_fee_usd                        true 0.067  substituted 1.060  cut 0.297  ok=True
  charge_review_policy   has_lookback_days                  true 0.157  substituted 1.498  cut 0.493  ok=True
  charge_r

In [5]:
# CELL 5 — policy-question bases. Admission = knowlytix's two-way functionality check (QuestionGenerator._validate,
# geometric=True): symbolic recompute AND rank-first among non-asserted tails AND inside the relation's cap_radius.
from gmsjourney.bases.kb import mine_kb_items
kb_items, kb_info = mine_kb_items(store)
for k in kb_items:
    print(f"{k.category:27s} {str(k.frame):70s} -> {k.answer!s:26s} [{k.recovery}]")
rec = kb_info["recovery"]
print("\nforged question (true answer, corrupted fact triple):", rec["forged"]["corrupted_triple"])
print("multi_hop ground truth (head, hop-1 value, tails):", rec["multi_hop_ground_truth"])
print("precondition, graph answers its own questions:", kb_info["precondition"])
for cat_, v in (rec.get("knowlytix_substrate") or {}).items():
    print(f"  substrate {cat_:15s}", v.get("admitted", v.get("skipped")), v.get("recovery_stats", ""))

exact_recall                ASK_KB(attr=fee, entity=standard_delivery, intent=value)               -> 0                          [two_way]
exact_recall                ASK_KB(attr=fee, entity=expedited_delivery, intent=value)              -> 15                         [two_way]
exact_recall                ASK_KB(attr=days, entity=standard_delivery, intent=value)              -> (5, 7)                     [two_way]
exact_recall                ASK_KB(attr=days, entity=expedited_delivery, intent=value)             -> (2, 2)                     [two_way]
exact_recall                ASK_KB(intent=limit, key=max_transactions_per_session)                 -> 4                          [two_way]
exact_recall                ASK_KB(intent=limit, key=lookback_days)                                -> 30                         [two_way]
exact_recall                ASK_KB(intent=limit, key=max_auth_attempts)                            -> 2                          [two_way]
fact_completion            

In [6]:
# CELL 6 — journey bases: constrained covering array + rule/edge/trigger/boundary closure.
from gmsjourney.oracle import PolicyOracle
from gmsjourney.bases.journey import JourneyBaseGenerator, to_item
from gmsjourney.journey import JourneySpec
from gmsjourney.pipeline import tags_for
oracle = PolicyOracle(store)
admitted = [k for k in kb_items if k.admission != "rejected"]
gen = JourneyBaseGenerator(oracle, kb_pool=[k.frame for k in admitted], seed=42)
cover, rep = gen.covering()
extra = gen.closure(cover, store, rep, extra_traces=[oracle.run(JourneySpec(n_txn=1, kb_mid="ask", kb_frames=(k.frame,))) for k in admitted])
specs = list({s.key(): s for s in cover + extra}.values())
bases = [to_item(sp, oracle, f"J{i:03d}") for i, sp in enumerate(specs)]
bases += [to_item(JourneySpec(n_txn=1, kb_mid="ask", kb_frames=(k.frame,)), oracle, k.qid, category=k.category, kb_item=k) for k in admitted]
print(len(specs), "journey bases +", len(admitted), "policy-question bases;",
      f"pairs {rep.pairs_covered}/{rep.pairs_required}, rules missing {rep.rules_missing}, edges missing {rep.edges_missing}")

54 journey bases + 27 policy-question bases; pairs 523/523, rules missing [], edges missing []


In [7]:
# CELL 7 — official catalogs (proofloop's bundled copy) + this bot's application layer, with level provenance.
from gmsjourney.catalog import Catalog, resolve_profile, validate, level_provenance_table
cat = Catalog.load()
print(cat.summary(), "| from:", cat.raw["official_origin"], "| contract violations:", validate(cat))
suite = resolve_profile(cat, "voice_full")
print(suite.summary())
for r in level_provenance_table(cat)[:6]:
    print(f"  {r['factor']:18s} {r['levels']}  unrealised: {list(r['unrealised'])}")

18 base categories in 5 families, 40 catalog factors in 10 groups (+2 custom), 13 factor groups, 9 profiles; application selects 16, excludes 23, moves 4 to the base | from: proofloop package | contract violations: []
suite[voice_full]: 18 bases x 16 factors, mode=embedded (0 factors dropped by applies_to/application)
  clarity            {'clear': 'Clear', 'indirect': 'Clear', 'downplayed': 'Misleading'}  unrealised: []
  entity_aliasing    {'canonical': 'Exact', 'synonym': 'Synonym', 'colloquial': 'Synonym'}  unrealised: ['Abbreviated']
  specificity        {'general': 'General', 'specific': 'Specific'}  unrealised: []
  numeric_format     {'digits': 'As_Shown', 'spoken': 'custom'}  unrealised: ['Percentage', 'Scientific']
  noise              {'none': 'None', 'typo': 'Typos', 'asr': 'custom'}  unrealised: ['OCR_Artifacts']
  disfluency         {'none': 'custom factor', 'fillers': 'custom factor', 'restart': 'custom factor'}  unrealised: []


In [8]:
# CELL 8 — design: our phi_p Sobol+refine (rank transform + swaps keep every level balanced) vs knowlytix DesignMatrix.
from gmsjourney.design.doe import compare_designs, compose, design_quality
ls = suite.level_specs(); n_runs = 6 * len(bases)
q = compare_designs(ls, n_runs)
if backend.available():
    from gmsjourney.pipeline import _kx_design_quality
    q["knowlytix DesignMatrix sobol+refine"] = _kx_design_quality(ls, n_runs, 42)
for k, v in q.items():
    print(f"{k:36s} log_phi_p={v['log_phi_p']:.3f}  min_dist={v['min_distance']:.3f}  worst marginal chi2 p={v['worst_marginal_chi2_p']:.4f}")
scenarios = compose(ls, bases, n_runs, 42, mode="embedded", assign="dispersed")

random                               log_phi_p=0.670  min_dist=0.526  worst marginal chi2 p=0.0183
balanced (proofloop)                 log_phi_p=0.989  min_dist=0.372  worst marginal chi2 p=1.0000
sobol                                log_phi_p=0.684  min_dist=0.526  worst marginal chi2 p=0.9817
sobol+refine                         log_phi_p=0.263  min_dist=0.832  worst marginal chi2 p=1.0000
knowlytix DesignMatrix sobol+refine  log_phi_p=0.325  min_dist=0.744  worst marginal chi2 p=0.0000


In [9]:
# CELL 9 — the proof: every enriched scenario is the same test as its base case.
from gmsjourney.enrich.realizers import Realizer
from gmsjourney.enrich.reader import FrameReader
from gmsjourney.enrich.invariance import InvarianceValidator
realizer, reader = Realizer(), FrameReader(store)
validator = InvarianceValidator(store, oracle, reader, realizer)
proofs = [validator.prove(s) for s in scenarios]
summ = validator.summarize(proofs, suite.factor_names)
print({k: summ[k] for k in ("scenarios", "acceptance_rate", "turns_checked", "L1_gt_identical", "L2_frames_equivalent", "L3_oracle_rederived")})
print("negative controls:", validator.negative_controls(scenarios, per_type=20)["_overall"])

{'scenarios': 486, 'acceptance_rate': 1.0, 'turns_checked': 4428, 'L1_gt_identical': 486, 'L2_frames_equivalent': 486, 'L3_oracle_rederived': 486}


negative controls: {'n': 342, 'detection_rate': 1.0}


### LLM enrichment arm (Microsoft model gateway), proved separately
The gateway token comes from `gcloud auth print-access-token`. Set `GMSJOURNEY_LLM_URL` to the exact chat-completions
URL (`{model}` is filled in) and, if needed, `GMSJOURNEY_CA_BUNDLE`. Check the connection once with `python tools/check_llm.py`. The LLM writes each
customer turn **blind** (meaning in plain words + base wording + style instructions; never the expected outcome).
A turn is used only if it passes:
- **G0 form** — one spoken line;
- **G1 facts** — no new figures or merchants;
- **G3 one clear decision**;
- **G2 meaning** — the rule reader, plus an independent LLM reader that never saw the base, recover the base meaning card in its dialog state.

The scenario must then pass the same L1/L3 test as the template arm. Negative controls are rerun on the LLM text,
and adversarial LLM rewrites must be rejected too. If the gateway cannot be reached, only cached generations are used.

In [ ]:
# CELL 9d — the LLM arm on the same design rows (paired with the template arm).
# A small population, proved correctly: candidates are tried (spread over the bases) until LLM_TARGET
# conversations pass every check. Asides are added verbatim around the model's line, and a turn the model
# cannot write correctly in its attempts falls back to the proved template turn (marked turn_source).
import os
from gmsjourney.enrich.llm import LLMSettings, build_llm_arm
from gmsjourney.pipeline import llm_pool, pick_llm_rows
# the Microsoft gateway: set the exact endpoint (and CA bundle) once, before LLMSettings is created
# os.environ["GMSJOURNEY_LLM_URL"] = "https://gaiapis.aexp.com/gcpapi/<your path>/completions"
# os.environ["GMSJOURNEY_CA_BUNDLE"] = "/etc/ssl/certs/ca-certificates.crt"
settings = LLMSettings(model="gpt-41", temperature=0.5, mode="live",      # mode="replay": cache only
                       log_responses=True, log_max_chars=400,              # print each live gateway reply
                       timeout=100, connect_timeout=50, max_retries=2)
llm_realizer, llm_validator, clients = build_llm_arm(store, oracle, reader, validator, settings, dual_read="gate",
                                                     max_attempts=3,            # attempts per turn, with feedback
                                                     asides="verbatim",         # | "paraphrase"
                                                     turn_fallback="template",  # | "none" (reject the scenario)
                                                     template_acts=("ASK_KB",)) # policy questions keep template wording
print("LLM callable:", clients["rephrase"].unavailable_reason() or "yes")
LLM_TARGET, LLM_WORKERS = 20, 8                          # accepted conversations wanted; concurrent gateway calls
# conversation rows only: the policy-question bases and the turn-limit rows padded with 15+ policy questions stay
# with the template arm (they test facts and the turn limit, not wording)
pool = pick_llm_rows(llm_pool(scenarios, ("policy_questions", "turn_padding")), 1.0, None, 49, ordered=True)
print(f"LLM candidate pool: {len(pool)} of {len(scenarios)} design rows")
llm_scenarios, llm_proofs = llm_validator.prove_target(pool, LLM_TARGET, max_candidates=3 * LLM_TARGET,
                                                       workers=LLM_WORKERS)
ls_ = llm_validator.summarize_llm(llm_proofs, suite.factor_names)
print({k: ls_[k] for k in ("scenarios", "accepted", "rejected_meaning_changed", "not_generated_network",
                           "turns_written_by_llm", "turns_template_fallback", "turns_template_by_design",
                           "llm_turn_share_of_accepted",
                           "mean_attempts_per_turn", "dual_read", "gate_failures_by_attempt")})
print("why turns fell back to the template:", ls_["fallback_reasons"])
if ls_.get("llm_errors"):                                # every call failed: show why (endpoint, HTTP status, TLS ...)
    print("LLM call errors:", ls_["llm_errors"])
    print("rephrase client:", {k: clients["rephrase"].report().get(k) for k in ("endpoint", "last_error", "adaptations")})
# write the per-turn LLM file (the campaign writes it too): every candidate, both readers, every gate
from gmsjourney.enrich.llm.export import write_llm_exports
print(write_llm_exports("output/llm_cell9d", llm_proofs, None), "-> output/llm_cell9d/llm_enrichment_turns.csv")
# why attempts were rejected (per check, per meaning card and state, per factor level, with examples):
#   !python tools/llm_rejections.py output/llm_cell9d/llm_enrichment_turns.csv
llm_validator.prefetch_controls(llm_proofs, llm_scenarios, per_type=20, workers=LLM_WORKERS)
llm_controls = llm_validator.negative_controls(llm_proofs, llm_scenarios, per_type=20, adversarial=10)
print("controls on LLM text:", llm_controls["_overall"], "| adversarial:", {k: llm_controls["_adversarial"][k]
      for k in ("n", "rejected", "effective", "rejection_rate_when_meaning_changed")})
# one accepted example, turn by turn
p = next((p for p in llm_proofs if p.accepted), None)
if p:
    for t in p.turns:
        print(f"[{t['state']}] base: {t['base_text']}\n      template: {t['template_text']}\n      LLM: {t['enriched_text']}"
              f"\n      rule reader: {t['read_back']} | independent reader: {t['dual_read']} | gates {t['gates']}")


### Inspect a case: seed vs enriched, canonical form and Head / Relation / Tail
Set `BASE` (a base id such as `"J015"` or `"kb-exact_recall-01"`, or an index), `VARIANT` (which enriched variant of that base) and
`TURN` (a customer turn index, or `None` for all turns). The first table compares the seed text and the enriched text
with their canonical frames; the second shows every turn as Head / Relation / Tail triples for the seed case and the
enriched case side by side (`match` = same tail), plus the store facts that ground each slot value.

The canonical form is **state-aware**: each customer turn is read in the dialog state it answers (the bot act it
replies to). A bare "yes" is RECOGNIZED after "do you recognise this charge?" (PRESENT_TXN) but CONFIRM after "shall
I go ahead?" (CONFIRM_ACTION); a card must be admissible in its state (the policy graph says which meaning cards
each state accepts). With `grounding=True` the tables also show the **state policy** rows: stage, rules, limits and
accepted cards for the state.

In [10]:
# CELL 9b — inspect one case: canonical seed vs enriched, and H / R / T per state and turn.
from IPython.display import display
from gmsjourney.inspect_cases import show_case, compare_case, case_hrt, pick, variants_of

BASE, VARIANT, TURN = "J015", 0, None        # e.g. BASE="kb-exact_recall-01" for a policy-question case; TURN=2 for one turn
print("base ids:", ", ".join(b.qid for b in bases))
print(f"{BASE}: {len(variants_of(scenarios, BASE if isinstance(BASE, str) else bases[BASE].qid))} enriched variants")
proof = show_case(validator, bases, scenarios, BASE, VARIANT, TURN, grounding=True, display=display)

# long form (one row per triple, case = seed / enriched / store fact):
item, scn = pick(bases, scenarios, BASE, VARIANT)
display(case_hrt(validator, scn, TURN, grounding=False, proof=proof, wide=False))

# optional dropdowns (if ipywidgets is installed)
try:
    import ipywidgets as w
    def _view(base, variant, turn):
        show_case(validator, bases, scenarios, base, variant, None if turn < 0 else turn, display=display)
    w.interact(_view, base=w.Dropdown(options=[b.qid for b in bases], value=BASE if isinstance(BASE, str) else bases[BASE].qid),
               variant=w.IntSlider(min=0, max=11, value=VARIANT), turn=w.IntSlider(min=-1, max=20, value=-1))
except ImportError:
    pass

base ids: J000, J001, J002, J003, J004, J005, J006, J007, J008, J009, J010, J011, J012, J013, J014, J015, J016, J017, J018, J019, J020, J021, J022, J023, J024, J025, J026, J027, J028, J029, J030, J031, J032, J033, J034, J035, J036, J037, J038, J039, J040, J041, J042, J043, J044, J045, J046, J047, J048, J049, J050, J051, J052, J053, kb-exact_recall-00, kb-exact_recall-01, kb-exact_recall-02, kb-exact_recall-03, kb-exact_recall-04, kb-exact_recall-05, kb-exact_recall-06, kb-fact_completion-00, kb-plausibility-00, kb-plausibility-01, kb-plausibility-02, kb-plausibility-03, kb-plausibility-04, kb-absence_detection-00, kb-ranking-00, kb-comparative_discrimination-00, kb-counting-00, kb-cross_reference-00, kb-numeric_computation-00, kb-cross_table_aggregation-00, kb-multi_hop-00, kb-conditional_rule-00, kb-conditional_rule-01, kb-boundary-00, kb-boundary-01, kb-procedural_ordering-00, kb-governance_policy-00
J015: 6 enriched variants
seed case J015  ->  enriched variant J015~0000   (proof ac

,turn,state,seed_text,enriched_text,seed_canonical,enriched_canonical,same_meaning,surface_cosine
0,0,ASK_AUTH,The last four digits of my card are 4821.,"so, I already checked my uh, stateent this mor...","{""act"": ""PROVIDE_CREDENTIAL"", ""slots"": {""crede...","{""act"": ""PROVIDE_CREDENTIAL"", ""slots"": {""crede...",True,0.312
1,1,ASK_AUTH_RETRY,The last four digits of my card are 4821.,"uh, I already you know, chevked my okay so, st...","{""act"": ""PROVIDE_CREDENTIAL"", ""slots"": {""crede...","{""act"": ""PROVIDE_CREDENTIAL"", ""slots"": {""crede...",True,0.224
2,2,PRESENT_TXN,I'm not sure about that charge.,"you know, I akready like, chrcked my so, staye...","{""act"": ""INFORM_DISPOSITION"", ""slots"": {""dispo...","{""act"": ""INFORM_DISPOSITION"", ""slots"": {""dispo...",True,0.167
3,3,PROVIDE_CONTEXT,"Yes, I recognize that charge.","like, I already checked my ststement uh, this ...","{""act"": ""INFORM_DISPOSITION"", ""slots"": {""dispo...","{""act"": ""INFORM_DISPOSITION"", ""slots"": {""dispo...",True,0.188



-- Head / Relation / Tail (seed vs enriched) --


,turn,state,seed_head,relation,seed_tail,enriched_head,enriched_tail,match
0,0,ASK_AUTH,J015/t0,in_state,ASK_AUTH,J015~0000/t0,ASK_AUTH,True
1,0,ASK_AUTH,J015/t0,has_act,PROVIDE_CREDENTIAL,J015~0000/t0,PROVIDE_CREDENTIAL,True
2,0,ASK_AUTH,J015/t0,has_credential,4821,J015~0000/t0,4821,True
3,0,ASK_AUTH,ASK_AUTH,in_stage,authenticate,ASK_AUTH,authenticate,state policy
4,0,ASK_AUTH,ASK_AUTH,applies_rule,"R2, R3, R5, R20, R22, R23, R25, R28",ASK_AUTH,"R2, R3, R5, R20, R22, R23, R25, R28",state policy
5,0,ASK_AUTH,ASK_AUTH,limited_by,"max_auth_attempts=2, max_turns=20",ASK_AUTH,"max_auth_attempts=2, max_turns=20",state policy
6,0,ASK_AUTH,ASK_AUTH,accepts_act,"PROVIDE_CREDENTIAL, REQUEST_HUMAN, ASK_KB, UNC...",ASK_AUTH,"PROVIDE_CREDENTIAL, REQUEST_HUMAN, ASK_KB, UNC...",state policy
7,1,ASK_AUTH_RETRY,J015/t1,in_state,ASK_AUTH_RETRY,J015~0000/t1,ASK_AUTH_RETRY,True
8,1,ASK_AUTH_RETRY,J015/t1,has_act,PROVIDE_CREDENTIAL,J015~0000/t1,PROVIDE_CREDENTIAL,True
9,1,ASK_AUTH_RETRY,J015/t1,has_credential,4821,J015~0000/t1,4821,True


,case,turn,state,head,relation,tail
0,seed,0,ASK_AUTH,J015/t0,in_state,ASK_AUTH
1,seed,0,ASK_AUTH,J015/t0,has_act,PROVIDE_CREDENTIAL
2,seed,0,ASK_AUTH,J015/t0,has_credential,4821
3,enriched,0,ASK_AUTH,J015~0000/t0,in_state,ASK_AUTH
4,enriched,0,ASK_AUTH,J015~0000/t0,has_act,PROVIDE_CREDENTIAL
5,enriched,0,ASK_AUTH,J015~0000/t0,has_credential,4821
6,seed,1,ASK_AUTH_RETRY,J015/t1,in_state,ASK_AUTH_RETRY
7,seed,1,ASK_AUTH_RETRY,J015/t1,has_act,PROVIDE_CREDENTIAL
8,seed,1,ASK_AUTH_RETRY,J015/t1,has_credential,4821
9,enriched,1,ASK_AUTH_RETRY,J015~0000/t1,in_state,ASK_AUTH_RETRY


In [11]:
# CELL 9c — the state-aware ontology mapper on its own: the same words, read in different dialog states.
from gmsjourney.enrich.reader import FrameReader
from gmsjourney.ontology import Ontology
ont, rd = Ontology.from_store(store), FrameReader(store)
rows = []
for text in ["Yes.", "No.", "My card ends 4821 and that one is not mine.", "The last four digits of my card are 4821."]:
    for st in ["ASK_AUTH", "PRESENT_TXN", "CONFIRM_ACTION", "ASK_MORE_HELP"]:
        c = ont.card(rd.read(text, st), st)
        rows.append({"text": text, "state": st, "act": c["act"], "slots": c["slots"], "admissible": c["admissible"],
                     "why_not": c.get("why_not", "")})
display(pd.DataFrame(rows))

,text,state,act,slots,admissible,why_not
0,Yes.,ASK_AUTH,UNCLEAR,{},True,
1,Yes.,PRESENT_TXN,INFORM_DISPOSITION,{'disposition': 'RECOGNIZED'},True,
2,Yes.,CONFIRM_ACTION,CONFIRM,{},True,
3,Yes.,ASK_MORE_HELP,UNCLEAR,{},True,
4,No.,ASK_AUTH,UNCLEAR,{},True,
5,No.,PRESENT_TXN,INFORM_DISPOSITION,{'disposition': 'UNRECOGNIZED'},True,
6,No.,CONFIRM_ACTION,DENY,{},True,
7,No.,ASK_MORE_HELP,END,{},True,
8,My card ends 4821 and that one is not mine.,ASK_AUTH,PROVIDE_CREDENTIAL,{'credential': '4821'},True,
9,My card ends 4821 and that one is not mine.,PRESENT_TXN,INFORM_DISPOSITION,{'disposition': 'UNRECOGNIZED'},True,


In [12]:
# CELL 10 — runtime gates (Appendix C): plausibility gate on `enables` calibrated on the policy graph;
# forgeloop's GMSPlausibilityGate scores transitions on the knowlytix store.
from gmsjourney.governance.calibrate import calibrate_plausibility, calibrate_plan, gate_probe_sweep
from gmsjourney.governance.monitor import GovernanceConfig
cal = calibrate_plausibility(store)
pc = calibrate_plan(store, [[c["tool"] for c in oracle.run(sp).tool_log] for sp in specs])
print("theta =", cal["theta"], cal["regulated"], "AUC", cal["auc"])
print("plan gate:", pc["criterion"], pc["drift_budget"], "| probe sweep:", gate_probe_sweep(store, cal["theta"])["confusion"])
gcfg = GovernanceConfig(theta=cal["theta"], drift_budget=pc["drift_budget"], plan_criterion=pc["criterion"])

theta = 0.9257 {'theta': 0.9257, 'false_admits': 0, 'false_rejects': 0, 'accuracy': 1.0} AUC 1.0
plan gate: mean 0.6671 | probe sweep: {'tp': 15, 'fp': 0, 'tn': 27, 'fn': 0}


In [13]:
# CELL 11 — run the bot under the runtime monitor; attribute failures (knowlytix DOEAnalyzer + per component).
from gmsjourney.core import Scenario
from gmsjourney.sut.adapter import JourneyRunner, make_reference_factory, CallableBotAdapter
from gmsjourney.evaluation.runner import run, summary
from gmsjourney.evaluation import attribution
factory = make_reference_factory(store)     # replace with your bot's factory (see README: Testing your real bot)
runner = JourneyRunner(bot_factory=factory, governance=gcfg, store=store, realizer=realizer, reader=reader)
base_rows = run([Scenario(f"{b.qid}~base", b, {}, "base") for b in bases], runner, oracle.limits, store.workflow)
rows = run(scenarios, runner, oracle.limits, store.workflow, validator)
print("base:", summary(base_rows)["accuracy"], "| enriched:", summary(rows)["accuracy"])
diag = attribution.diagnose(rows, suite.factor_names)
if backend.available():
    from gmsjourney.reuse.attribution import kx_attribution
    for t in sorted(kx_attribution(rows, suite.factor_names)["logistic"], key=lambda t: t["p_value"])[:5]:
        print(f"  {t['factor']:20s} p={t['p_value']:.2e}  significant={t.get('significant_adj', t.get('significant'))}")

base: 0.9012345679012346 | enriched: 0.5


  noise                p=9.94e-12  significant=True
  numeric_format       p=1.89e-05  significant=True
  disfluency           p=2.16e-03  significant=True
  history              p=6.40e-02  significant=False
  self_correction      p=2.38e-01  significant=False


In [ ]:
# CELL 11c — LLM-arm runs, paired comparison, budgets, severity and the tiered release gate.
from gmsjourney.evaluation.constraints import ConstraintBudget
from gmsjourney.evaluation.severity import release_gate
from gmsjourney.pipeline import _arm_comparison
runner2 = JourneyRunner(bot_factory=factory, governance=gcfg, store=store, realizer=realizer, reader=reader,
                        arm_realizers={"llm": llm_realizer}, budgets=ConstraintBudget(max_bot_latency_ms=2500, max_bot_words=60))
rows_t = run(scenarios, runner2, oracle.limits, store.workflow, validator)
rows_l = run(llm_scenarios, runner2, oracle.limits, store.workflow, llm_validator)
print("template:", summary(rows_t)["accuracy"], "| LLM:", summary(rows_l)["accuracy"] if rows_l else None)
A = _arm_comparison(rows_t, rows_l)
print({k: v for k, v in A.items() if k != "pairs"})
for name, rr in (("template", rows_t), ("llm", rows_l)):
    g = release_gate(rr)
    print(name, "release gate:", g["passed"], {t: (v["passed"], v["reasons"]) for t, v in g["tiers"].items()})


In [14]:
# CELL 11b — turn by turn: what the oracle says the bot should do at each customer turn (tools with arguments and
# results, bot acts) next to what the bot actually did (tools, acts, text). Grading is on the whole run (outcome,
# per-stage decisions, admissibility); this view shows where and how a run went wrong.
from gmsjourney.turn_compare import show_turns, turn_by_turn
CASE = "J003~base"                         # any base ("J003~base") or enriched ("J003~0007") case id
all_scns = {s.sid: s for s in [Scenario(f"{b.qid}~base", b, {}, "base") for b in bases] + scenarios}
pd.set_option("display.max_colwidth", 80)
df, s = show_turns(runner, oracle, all_scns[CASE], validator=validator, display=display)
# the first failed enriched run, for comparison
failed = next(sid for sid in (r["sid"] for r in rows if not r["correct"]))
df2, s2 = show_turns(runner, oracle, all_scns[failed], validator=validator, display=display)

J003~base: outcome CORRECT | turns matching 4/4 | first divergence: None | stages wrong: -
expected outcome: {'action': None, 'end': 'TRANSFERRED', 'recorded': [['T1', 'UNRECOGNIZED']], 'transfer_reason': 'card_options_unavailable'}
actual outcome:   {'action': None, 'end': 'TRANSFERRED', 'recorded': [['T1', 'UNRECOGNIZED']], 'transfer_reason': 'card_options_unavailable'}


,turn,state,customer_text,expected_tools,actual_tools,expected_bot,actual_bot_acts,turn_match
0,0,ASK_AUTH,The last four digits of my card are 4821.,authenticate_customer(credential=4821) -> FAILED,authenticate_customer(credential=4821) -> FAILED,ASK_AUTH_RETRY,ASK_AUTH_RETRY,True
1,1,ASK_AUTH_RETRY,The last four digits of my card are 4821.,authenticate_customer(credential=4821) -> AUTHENTICATED ; get_review_transac...,authenticate_customer(credential=4821) -> AUTHENTICATED ; get_review_transac...,PRESENT_TXN(T1: Amazon $45.67),PRESENT_TXN(T1: Amazon $45.67),True
2,2,PRESENT_TXN,"Hmm, sorry, what was that?",-,-,ASK_CLARIFY(T1: Amazon $45.67),ASK_CLARIFY(T1: Amazon $45.67) ; PRESENT_TXN(T1: Amazon $45.67),True
3,3,ASK_CLARIFY,"No, I don't recognize that charge. Please also replace my card with standard...","record_charge_response(transaction_id=T1, disposition=UNRECOGNIZED) -> SUCCE...","record_charge_response(transaction_id=T1, disposition=UNRECOGNIZED) -> SUCCE...",SUMMARIZE(T1=UNRECOGNIZED) ; TRANSFER(card_options_unavailable),CONFIRM_RECORDED(T1: Amazon $45.67; UNRECOGNIZED) ; SUMMARIZE(T1=UNRECOGNIZE...,True


J046~0001: outcome WRONG | turns matching 0/4 | first divergence: 0 | stages wrong: ['authenticate', 'retrieve', 'present', 'kb', 'capture', 'record', 'transfer']
expected outcome: {'action': None, 'end': 'TRANSFERRED', 'recorded': [], 'transfer_reason': 'record_failed'}
actual outcome:   {'action': None, 'end': 'TRANSFERRED', 'recorded': [], 'transfer_reason': 'ambiguity_persists'}


,turn,state,customer_text,expected_tools,actual_tools,expected_bot,actual_bot_acts,turn_match
0,0,ASK_AUTH,"okay so, i was just on the phone about my mortgage earlier, anyway. the last...",authenticate_customer(credential=4821) -> AUTHENTICATED ; get_review_transac...,-,PRESENT_TXN(T1: Amazon $45.67),ASK_CLARIFY ; ASK_AUTH,False
1,1,PRESENT_TXN,"um, i was just on hmm, the um, phone about my mortgage earlier, anyway. the ...",-,transfer_to_human(reason=ambiguity_persists) -> TRANSFERRED,ANSWER_KB(window_check -> false) ; PRESENT_TXN(T1: Amazon $45.67),TRANSFER(ambiguity_persists),False
2,2,PRESENT_TXN,"(expected path) hmm, i was just on the phone about my mortgage earlier, anyw...",-,(no turn),PROVIDE_CONTEXT(T1: Amazon $45.67),-,False
3,3,PROVIDE_CONTEXT,"(expected path) uh, i was just so, on hmm, the phone about my mortgage earli...","record_charge_response(transaction_id=T1, disposition=UNSURE) -> FAILED ; tr...",(no turn),TRANSFER(record_failed),-,False


In [15]:
# CELL 12 — resilience: error / timeout / stale / plausible-wrong faults per tool, scored against the oracle.
from gmsjourney.evaluation import resilience
probe_scns = [Scenario(f"{b.qid}~base", b, {}, "base") for b in bases
              if b.metadata["relevant"].get("n_txn") not in ("N/A", 0) and not b.metadata.get("kb_qid")]
for r in resilience.probe(runner, oracle, probe_scns):
    print(f"{r['tool']:24s} {r['kind']:15s} reached={r['reached']:3d} silent={r['silent']} detection={r['detection_rate']:.2f}")

authenticate_customer    error           reached= 47 silent=0 detection=1.00
authenticate_customer    timeout         reached= 47 silent=0 detection=1.00
authenticate_customer    stale           reached= 18 silent=0 detection=1.00
authenticate_customer    plausible_wrong reached= 47 silent=0 detection=1.00
get_review_transactions  error           reached= 47 silent=0 detection=1.00
get_review_transactions  timeout         reached= 47 silent=0 detection=1.00
record_charge_response   error           reached= 33 silent=0 detection=1.00
record_charge_response   timeout         reached= 33 silent=0 detection=1.00
record_charge_response   stale           reached= 10 silent=0 detection=1.00
record_charge_response   plausible_wrong reached= 33 silent=0 detection=1.00
get_card_action_options  error           reached= 17 silent=0 detection=1.00
get_card_action_options  timeout         reached= 17 silent=0 detection=1.00
get_card_action_options  plausible_wrong reached= 17 silent=0 detection=1.00

In [16]:
# CELL 13 — mutation analysis: 17 seeded defects (incl. kb_fabricate_unknown, caught by the hallucination probes); caught by which suite, and flagged live by the runtime monitor.
# Same suites as the pipeline (minus the random comparison suite); tool-fault probes let the monitor see fault-path defects.
from gmsjourney.evaluation import mutation
base_scns = [Scenario(f"{b.qid}~base", b, {}, "base") for b in bases]
suites = {"covering_base": [s for s in base_scns if not s.base.metadata.get("kb_qid")],
          "kb_bases": [s for s in base_scns if s.base.metadata.get("kb_qid")],
          "enriched_design": scenarios}
mres = mutation.mutation_analysis(store, {"store": store, "realizer": realizer, "reader": reader}, suites,
                                  oracle.limits, store.workflow, validator, fault_scenarios=probe_scns, governance=gcfg)
print(mres["mutation_score"], "| runtime monitor catch rate:", mres.get("runtime_monitor_catch_rate"))

{'covering_base': 0.75, 'kb_bases': 0.25, 'enriched_design': 0.9375, 'fault_probes': 0.25, 'all_suites_combined': 1.0} | runtime monitor catch rate: 1.0


In [17]:
# CELL 14 — training corpora with proofloop's emitters (label-invariant classifier corpus; contract-checked drafts).
# This step needs the open proofloop package; without it the corpus emission is skipped (the built-in backend
# emits no training corpora — see the pipeline's step 11, which reports the same skip).
try:
    from gmsjourney.reuse.sft import classifier_corpus, draft_corpus
    from gmsjourney.ontology import Ontology
    from gmsjourney.sut.reference_bot import DEFAULT_CONFIG, KBAnswerer
    cls = classifier_corpus(proofs, reader, Ontology.from_store(store))
    print({k: v for k, v in cls.items() if k != "records"})
    ans = KBAnswerer([s for _, s in store.kb_sentences], dict(DEFAULT_CONFIG), {"kb_first_amount"})
    d = draft_corpus(store, admitted, lambda fr, q: ans.answer(q), realizer)
    print("draft corpus with a defective answerer: kept", d["kept"], "dropped", d["dropped"])
except ImportError as _e:
    print(f"training corpora skipped (proofloop not installed: {_e})")

{'n': 4428, 'dropped_by_grounding_filter': 0, 'schema': ['_factors', '_seed', 'label', 'message'], 'seeds': 738, 'one_label_per_seed': True, 'distinct_messages': 3688}
draft corpus with a defective answerer: kept 25 dropped 2


In [ ]:
# CELL 15 — or everything in one call, and open the HTML report.
# Reloads the GMS built in CELL 3 (the first build is the long step); the report, CSVs and
# training corpora land in output/campaign_out/.
# 0.4 options: kx_dedup_mode (off|audit|accept), kx_v_sft_mode (rotation|full|ab), decision_rule (strict|quorum),
# budgets, and the LLM arm: llm_arm=True, llm_mode ("live"|"replay"), llm_share, llm_dual_read ("gate"|"report"|"off").
# The LLM arm is the slow part (every rewritten row = ~2 gateway calls per turn, more on retries): keep it to a
# sample with llm_max_scenarios; the template arm still covers every design row.
from gmsjourney.pipeline import CampaignConfig, run_campaign
from gmsjourney.enrich.llm import LLMSettings
llm_settings = LLMSettings(model="gpt-41", temperature=0.5, mode="live",
                           log_responses=True, log_max_chars=400,
                           timeout=100, connect_timeout=50, max_retries=2)
R = run_campaign(CampaignConfig(
    out_dir="output/campaign_out",
    llm_arm=True, llm_settings=llm_settings,
    llm_target_accepted=24,         # keep proving candidates (round-robin over bases) until 24 are accepted
    llm_max_candidates=72,          # ... trying at most 72 design rows
    llm_max_attempts=3,             # attempts per turn, each retry told why the last one was rejected
    llm_asides="verbatim",          # template asides added around the LLM's line (no smuggled decision cues)
    llm_turn_fallback="template",   # a turn the LLM cannot write correctly -> the proved template turn
    llm_exclude=("policy_questions", "turn_padding"),   # rows left to the template arm
    llm_template_acts=("ASK_KB",),  # policy-question turns keep the proved template wording
    llm_workers=8,                  # concurrent gateway calls
    llm_dual_read="gate",           # independent LLM reader stays on: it is the invariance check of this arm
    llm_controls_per_type=10,       # controls on LLM text (each = 1 extract call)
    llm_adversarial_controls=6))    # adversarial rewrites (each = 1-2 calls)
from IPython.display import HTML, display
display(HTML(open("output/campaign_out/report.html").read()))

In [ ]:
# CELL 16 — the exported test suite and the trained GMS's canonicals.
import json
ds = "output/campaign_out/dataset"
cases = pd.read_csv(f"{ds}/test_cases.csv")
turns = pd.read_csv(f"{ds}/test_turns.csv")
print(R["dataset"])
display(cases[["case_id", "case_type", "arm", "category", "policy_question", "enrichment_changed", "proof_accepted"]].head(8))
print("cases by arm:", cases.groupby(["arm", "proof_accepted"]).size().to_dict())   # base / template / llm
display(turns[["case_id", "turn", "state", "state_rules", "meaning_card", "base_text", "enriched_text", "equivalent"]].head(8))
print(json.loads(cases.loc[cases.case_type == "enriched", "trajectory"].iloc[0])[:6])
canon = json.load(open("output/campaign_out/gms_canonicals.json"))
print(canon["summary"])
display(pd.DataFrame([{"value_set": vs, "canonical": m["canonical"], "aliases": len(m.get("aliases", [])),
                       "alias_agreement": m.get("alias_agreement"), "member_of_inside": m.get("member_of_inside")}
                      for vs, d in canon["value_sets"].items() for m in d["members"]]))

### The four documents in `docs/`

One call builds all four from the run above — the answer key for every case, the annotated listing of
the three invariance-critical source files, the walkthrough of the twelve steps, and the proof. The two
written pages are filled from `tools/templates/*.html` with **this** run's figures (the tiles and the
summary blocks follow whatever campaign you just ran); the other two are generated from the store and
the sources. Edit the templates, never `docs/*.html`.

In [1]:
# CELL 17 — the four documents in docs/: the answer key for every case, the annotated listing of the
# three invariance-critical files, the walkthrough of the twelve steps, and the proof.
# They are built from the newest completed campaign under output/ (cells 3-21 alone write no report,
# so run CELL 15 first, or point --campaign at any directory that has a report.json).
import subprocess, sys
from pathlib import Path
_done = sorted(Path("output").glob("*/report.json"), key=lambda p: p.stat().st_mtime, reverse=True)
_campaign = _done[0].parent if _done else Path("output/campaign_out")
print(f"building the four documents from {_campaign}")
_r = subprocess.run([sys.executable, "tools/build_docs.py", "--campaign", str(_campaign)],
                    capture_output=True, text=True)
print(_r.stdout or _r.stderr)
for _f in sorted(Path("docs").glob("*.html")):
    print(f"  {_f}  {_f.stat().st_size // 1024} KB")

building the four documents from output\campaign_llm


  answer_sheet.py …
    ok: wrote docs\ANSWER_SHEET.html (1261 KB)
  build_annotated.py …
    ok: wrote docs\ANNOTATED.html (171 KB)
  EXPLAINER.html: filled from tools/templates/EXPLAINER.html
  INVARIANCE.html: filled from tools/templates/INVARIANCE.html

4 document(s) in docs/ — from the run in output/campaign_llm (85 scenarios, 90.6% → 49.4%)

  docs\ANNOTATED.html  171 KB
  docs\ANSWER_SHEET.html  1281 KB
  docs\EXPLAINER.html  69 KB
  docs\INVARIANCE.html  36 KB
